<a href="https://colab.research.google.com/github/nithin12342/phase2/blob/main/ml_pipeline/h5_omnifusion/notebooks/H5_OmniFusion_Training_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount Drive first
from google.colab import drive
drive.mount('/content/drive')

# Then run the 5-fold Nano training
!python /content/phase2/ml_pipeline/h5_omnifusion/scripts/run_all_folds.py

# 🧠 H5-OmniFusion Training Notebook

**Purpose:** Train the H5-OmniFusion multimodal depression detection model using pre-extracted H5 features.

**New in v2:** More stable training with reduced learning rate and conservative focal loss parameters.

---

## Training Configuration
- **Model:** H5OmniFusion (~ 34000 parameters)
- **Optimizer:** AdamW with OneCycleLR, LR=3e-5
- **Loss:** Focal Loss (α=0.75, γ=2.0) with PHQ regression
- **Cross-Validation:** 5-Fold Stratified

## Step 1: Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

Mounted at /content/drive
✅ Google Drive mounted


## Step 2: Clone/Update Repository

In [3]:
import os

REPO_URL = 'https://github.com/nithin12342/phase2.git'
REPO_DIR = '/content/phase2'

if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    !git pull origin main
else:
    !git clone {REPO_URL} {REPO_DIR}

print(f'✅ Repository ready: {REPO_DIR}')

Cloning into '/content/phase2'...
remote: Enumerating objects: 1447, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 1447 (delta 7), reused 15 (delta 7), pack-reused 1423 (from 1)
Receiving objects: 100% (1447/1447), 13.56 MiB | 20.92 MiB/s, done.
Resolving deltas: 100% (753/753), done.
✅ Repository ready: /content/phase2


## Step 3: Install Dependencies

In [4]:
import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

%pip install -q transformers>=4.36.0 einops h5py tqdm scikit-learn pandas tabulate

print('✅ Dependencies installed')

PyTorch: 2.9.0+cu126, CUDA: True
✅ Dependencies installed


## Step 4: Configure Paths

**⚠️ IMPORTANT:** Update these paths to match your Google Drive structure!

In [5]:
import sys
import os

# Add project paths
PROJECT_ROOT = '/content/phase2'
ML_PIPELINE = f'{PROJECT_ROOT}/ml_pipeline/h5_omnifusion'

sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, ML_PIPELINE)
sys.path.insert(0, f'{ML_PIPELINE}/src')

# ╔═══════════════════════════════════════════════════════════════╗
# ║  DATA PATHS - MODIFY AS NEEDED                               ║
# ╚═══════════════════════════════════════════════════════════════╝

DATA_ROOT = '/content/drive/MyDrive/DAIC-WOZ_Datasets'
H5_OUTPUT_DIR = f'{DATA_ROOT}/H5_OmniFusion_Output'
LABELS_CSV = f'{DATA_ROOT}/merged_labels.csv'
CHECKPOINT_DIR = f'{DATA_ROOT}/H5_Training_Checkpoints'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f'📂 H5 Features: {H5_OUTPUT_DIR}')
print(f'📋 Labels: {LABELS_CSV}')
print(f'💾 Checkpoints: {CHECKPOINT_DIR}')

# Verify paths
if os.path.exists(H5_OUTPUT_DIR):
    datasets = [d for d in os.listdir(H5_OUTPUT_DIR) if os.path.isdir(f'{H5_OUTPUT_DIR}/{d}')]
    print(f'\n✅ Found {len(datasets)} dataset folders')
    for ds in datasets:
        h5_count = len([f for f in os.listdir(f'{H5_OUTPUT_DIR}/{ds}') if f.endswith('.h5')])
        print(f'   - {ds}: {h5_count} H5 files')
else:
    print(f'⚠️ H5 directory not found!')

if os.path.exists(LABELS_CSV):
    import pandas as pd
    labels_df = pd.read_csv(LABELS_CSV)
    print(f'\n✅ Labels: {len(labels_df)} participants')
else:
    print(f'⚠️ Labels CSV not found!')

📂 H5 Features: /content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output
📋 Labels: /content/drive/MyDrive/DAIC-WOZ_Datasets/merged_labels.csv
💾 Checkpoints: /content/drive/MyDrive/DAIC-WOZ_Datasets/H5_Training_Checkpoints

✅ Found 4 dataset folders
   - DAIC-WOZ: 189 H5 files
   - Extended-DAIC: 86 H5 files
   - EATD-Corpus: 83 H5 files
   - training_results_gemini: 0 H5 files

✅ Labels: 219 participants


## Step 5: Training Configuration (Stable Settings)

In [6]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║  STABLE TRAINING CONFIGURATION                               ║
# ╚═══════════════════════════════════════════════════════════════╝

TRAINING_CONFIG = {
    'tier': 'micro',
    'epochs': 100,
    'batch_size': 8,
    'learning_rate': 3e-5,    # Reduced for stability (was 1e-4)
    'n_folds': 5,
    'current_fold': 0,
    'patience': 20,           # More patience for slow convergence

    # Paths
    'data_dir': H5_OUTPUT_DIR,
    'labels_csv': LABELS_CSV,
    'checkpoint_dir': CHECKPOINT_DIR,
}

print('📋 Training Configuration:')
for k, v in TRAINING_CONFIG.items():
    print(f'   {k}: {v}')

📋 Training Configuration:
   tier: standard
   epochs: 100
   batch_size: 8
   learning_rate: 3e-05
   n_folds: 5
   current_fold: 0
   patience: 20
   data_dir: /content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output
   labels_csv: /content/drive/MyDrive/DAIC-WOZ_Datasets/merged_labels.csv
   checkpoint_dir: /content/drive/MyDrive/DAIC-WOZ_Datasets/H5_Training_Checkpoints


## Step 6: Check for Existing Checkpoints

In [7]:
import glob

def find_latest_checkpoint(checkpoint_dir, fold=None):
    if fold is not None:
        pattern = f'{checkpoint_dir}/h5_omnifusion_*_fold{fold}_*.pt'
    else:
        pattern = f'{checkpoint_dir}/*.pt'
    checkpoints = glob.glob(pattern)
    if not checkpoints:
        return None
    checkpoints.sort(key=os.path.getmtime, reverse=True)
    return checkpoints[0]

existing = glob.glob(f'{CHECKPOINT_DIR}/*.pt')
print(f'📂 Found {len(existing)} checkpoints')

resume_checkpoint = find_latest_checkpoint(CHECKPOINT_DIR, TRAINING_CONFIG['current_fold'])
if resume_checkpoint:
    print(f'🔄 Resume available: {os.path.basename(resume_checkpoint)}')
else:
    print(f'🆕 Starting fresh for fold {TRAINING_CONFIG["current_fold"]}')

📂 Found 5 checkpoints
🔄 Resume available: h5_omnifusion_standard_fold0_best.pt


## Step 7: Run Training

In [8]:
fold = TRAINING_CONFIG['current_fold']
n_folds = TRAINING_CONFIG['n_folds']

cmd = [
    'python', f'{ML_PIPELINE}/scripts/train.py',
    '--data_dir', TRAINING_CONFIG['data_dir'],
    '--labels_csv', TRAINING_CONFIG['labels_csv'],
    '--output_dir', TRAINING_CONFIG['checkpoint_dir'],
    '--tier', TRAINING_CONFIG['tier'],
    '--epochs', str(TRAINING_CONFIG['epochs']),
    '--batch_size', str(TRAINING_CONFIG['batch_size']),
    '--lr', str(TRAINING_CONFIG['learning_rate']),
    '--folds', str(n_folds),
    '--fold', str(fold),
    '--num_workers', '0',
]

if resume_checkpoint:
    cmd.extend(['--resume', resume_checkpoint])

print(f'🚀 Training Fold {fold + 1}/{n_folds}')
print(f'Command: {" ".join(cmd)}')
print('=' * 60)

!{' '.join(cmd)}

🚀 Training Fold 1/5
Command: python /content/phase2/ml_pipeline/h5_omnifusion/scripts/train.py --data_dir /content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output --labels_csv /content/drive/MyDrive/DAIC-WOZ_Datasets/merged_labels.csv --output_dir /content/drive/MyDrive/DAIC-WOZ_Datasets/H5_Training_Checkpoints --tier standard --epochs 100 --batch_size 8 --lr 3e-05 --folds 5 --fold 0 --num_workers 0 --resume /content/drive/MyDrive/DAIC-WOZ_Datasets/H5_Training_Checkpoints/h5_omnifusion_standard_fold0_best.pt
2026-02-02 04:22:04.270135: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770006124.291256    1435 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770006124.297365    1435 cuda_blas.cc:1407] Unable to register cuBLAS factory: Atte

## Step 8: Train All 5 Folds (Optional)

In [9]:
TRAIN_ALL_FOLDS = True  # Set True to train all folds

if TRAIN_ALL_FOLDS:
    for fold_idx in range(5):
        print(f'\n{"="*60}')
        print(f'🔄 FOLD {fold_idx + 1}/5')
        print(f'{"="*60}\n')

        resume = find_latest_checkpoint(CHECKPOINT_DIR, fold_idx)

        cmd = [
            'python', f'{ML_PIPELINE}/scripts/train.py',
            '--data_dir', TRAINING_CONFIG['data_dir'],
            '--labels_csv', TRAINING_CONFIG['labels_csv'],
            '--output_dir', TRAINING_CONFIG['checkpoint_dir'],
            '--tier', TRAINING_CONFIG['tier'],
            '--epochs', str(TRAINING_CONFIG['epochs']),
            '--batch_size', str(TRAINING_CONFIG['batch_size']),
            '--lr', str(TRAINING_CONFIG['learning_rate']),
            '--folds', '5',
            '--fold', str(fold_idx),
            '--num_workers', '0',
        ]

        if resume:
            cmd.extend(['--resume', resume])

        !{' '.join(cmd)}

    print('\n✅ All folds complete!')
else:
    print('Set TRAIN_ALL_FOLDS = True to train all folds')

Set TRAIN_ALL_FOLDS = True to train all folds


## Step 9: View Results

In [10]:
checkpoints = glob.glob(f'{CHECKPOINT_DIR}/*.pt')
checkpoints.sort(key=os.path.getmtime, reverse=True)

print(f'📂 Checkpoints: {len(checkpoints)}')
for ckpt in checkpoints[:10]:
    size_mb = os.path.getsize(ckpt) / (1024*1024)
    print(f'   - {os.path.basename(ckpt)} ({size_mb:.1f} MB)')

📂 Checkpoints: 5
   - h5_omnifusion_standard_fold1_best.pt (320.8 MB)
   - h5_omnifusion_standard_fold0_best.pt (320.8 MB)
   - h5_omnifusion_standard_fold4_best.pt (320.8 MB)
   - h5_omnifusion_standard_fold3_best.pt (320.8 MB)
   - h5_omnifusion_standard_fold2_best.pt (320.8 MB)
